# Previsao Climatica - pipeline WORCAP/INPE

Roda o pipeline completo (build_dataset -> train -> predict) usando a GPU e os dados da competicao ja montados pelo Kaggle em `/kaggle/input/`.

**Antes de rodar:**
1. Settings (painel direito) -> Accelerator -> **GPU T4 x2** (ou P100).
2. Add Data -> anexa o dataset da competicao (se ainda nao estiver anexado).
3. Add Data -> Upload -> sobe a pasta do repo local (`Previsao_Climatica/`, com `src/`, `pyproject.toml` etc) como um Dataset novo -- vira `/kaggle/input/<nome-que-voce-deu>`.
4. Ajusta `CODE_INPUT` na celula abaixo pro nome exato desse dataset.

In [ ]:
# confirma o nome exato da pasta do dataset da competicao
!ls /kaggle/input/competitions/

In [ ]:
# ajusta pro nome exato do dataset que voce subiu (Add Data -> Upload -> pasta do repo)
CODE_INPUT = "/kaggle/input/previsao-climatica-codigo"  # <-- troca pelo nome real do dataset

import os

src_dir = CODE_INPUT
if not os.path.exists(os.path.join(src_dir, "pyproject.toml")):
    # upload as vezes cria uma pasta extra aninhada (ex: .../Previsao_Climatica/Previsao_Climatica)
    subdirs = [d for d in os.listdir(src_dir) if os.path.isdir(os.path.join(src_dir, d))]
    nested = [d for d in subdirs if os.path.exists(os.path.join(src_dir, d, "pyproject.toml"))]
    if nested:
        src_dir = os.path.join(src_dir, nested[0])
        print(f"detectada pasta aninhada, usando: {src_dir}")

%cd /kaggle/working
!rm -rf Previsao_Climatica
!cp -r {src_dir} Previsao_Climatica
%cd /kaggle/working/Previsao_Climatica
!ls

In [ ]:
# a imagem do Kaggle ja vem com xgboost/pandas/xarray/torch com CUDA -- so garante versoes/pacotes que podem faltar
!pip install -q pyarrow h5netcdf netcdf4 xgboost --upgrade

In [ ]:
# ajusta o nome da pasta se for diferente do que apareceu no `ls /kaggle/input/competitions/` acima
COMPETITION_INPUT = "/kaggle/input/competitions/previsao-climatica-de-precipitacao-sobre-a-america-do-sul"

!ln -sfn {COMPETITION_INPUT} data
!ls -la data/

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Baseline climatologia (referencia a bater)

In [ ]:
!python -m src.baseline_climatology --split holdout

## 2. Build dataset (holdout) -- Kaggle tem bem mais RAM, pode tentar `--spatial-stride 1` (resolucao cheia) primeiro; se estourar, cai pra 2

In [ ]:
!python -m src.build_dataset --split holdout --spatial-stride 1

## 3. Treina no holdout (GPU) e ve o RMSE vs climatologia

In [ ]:
!python -m src.train --split holdout --device cuda

## 4. Build dataset full + treino full + predicao
So roda depois de confirmar que o holdout acima ficou bom (RMSE menor que a climatologia).

In [ ]:
!python -m src.build_dataset --split full --spatial-stride 1

In [ ]:
!python -m src.train --split full --device cuda --n-estimators 2000

In [ ]:
!mkdir -p /kaggle/working/submissions
!python -m src.predict --split full --output /kaggle/working/submission.csv

## 5. Submeter
`/kaggle/working/submission.csv` fica disponivel na aba **Output** do notebook -- da pra clicar em **Submit to Competition** direto dali, sem precisar baixar/subir manualmente.